In [1]:
import pandas as pd
import openai
from tqdm import tqdm
from ast import literal_eval
import threading
from queue import Queue
import json
from openai import OpenAI
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score

/Users/xuanyushi/Desktop/for_github/llm-evidence-triangulation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
%pip install scikit-learn
%pip install bert_score

# nc study

In [4]:
nc_input_df = pd.read_csv('../data/vali_data/nc_pmids_result.csv').drop(columns=['Unnamed: 0'])
#('/content/drive/MyDrive/PKU/杜老师/automated_triangulation/科技部验证/nc_pmids_result.csv').drop(columns=['Unnamed: 0'])

print(nc_input_df.shape,nc_input_df.columns)
nc_input_df['input'] = "Title: " + nc_input_df['article_title'] + '. Abstract: ' + nc_input_df['full_abs']
nc_input_df['pmid'] = nc_input_df['pmid'].astype(int).astype(str)
#nc_input_df  = nc_input_df.dropna(subset=['input'])
print(nc_input_df.shape,nc_input_df.columns)

input_df = nc_input_df[~nc_input_df['pmid'].isin(['8688759','17379620'])]
print(input_df['pmid'].nunique())

(78, 17) Index(['pmid', 'pub_year', 'pub_month', 'epub_year', 'epub_month', 'pub_types',
       'full_abs', 'article_title', 'background', 'method', 'result',
       'conclusion', 'label_result', 'label_conclusion', '# of comments',
       'all_mesh_terms', 'major_mesh_terms'],
      dtype='object')
(78, 18) Index(['pmid', 'pub_year', 'pub_month', 'epub_year', 'epub_month', 'pub_types',
       'full_abs', 'article_title', 'background', 'method', 'result',
       'conclusion', 'label_result', 'label_conclusion', '# of comments',
       'all_mesh_terms', 'major_mesh_terms', 'input'],
      dtype='object')
76


In [7]:
nc_vali_df = pd.read_excel('../data/vali_data/nc_figure1.xlsx').rename(columns={'Category':'category','Exposure':'exposure','Label':'label','Phenotype':'outcome_phenotype','Association':'association','Effect Size':'effect_size','PubMed_ID':'pmid'})
print(nc_vali_df.shape,nc_vali_df.columns,nc_vali_df['pmid'].nunique())
print(nc_vali_df['category'].unique())
print(nc_vali_df['outcome_phenotype'].unique())

nc_vali_df = nc_vali_df.dropna(subset=['pmid'])

nc_vali_df['pmid'] = nc_vali_df['pmid'].astype(int).astype(str)

nc_vali_df = nc_vali_df[nc_vali_df['pmid'].isin(input_df['pmid'])]
print(nc_vali_df.shape,nc_vali_df.columns,nc_vali_df['pmid'].nunique())

nc_vali_df['association'] = nc_vali_df['association'].replace({'neg':'negative','pos':'positive'})
nc_vali_df.sample(3)

(291, 7) Index(['category', 'exposure', 'label', 'outcome_phenotype', 'association',
       'effect_size', 'pmid'],
      dtype='object') 78
['Food/Diet' 'Lipid' 'Mineral' 'Vitamin' 'Carb' 'Alcohol' 'Fiber' 'Other'
 'Protein']
['CHD' 'MI' 'Hemorrhagic stroke' 'Ischemic stroke' 'Total stroke'
 'Ischemic heart disease' 'Nonfatal MI' 'Sudden cardiac death' 'CVD'
 'Peripheral artery disease' 'Coronary artery disease' 'Thrombotic stroke'
 'Hypertension' 'Atherothrombotic stroke' 'Cardioembolic stroke']
(249, 7) Index(['category', 'exposure', 'label', 'outcome_phenotype', 'association',
       'effect_size', 'pmid'],
      dtype='object') 76


,category,exposure,label,outcome_phenotype,association,effect_size,pmid
104,Food/Diet,tea,Tea,Total stroke,no,NaN,19221216
159,Other,plasma fetuin-A,Plasma fetuin-A,CHD,no,NaN,24963103
44,Vitamin,dietary vitamin C,D-vitamin C,CHD,no,NaN,12875759


In [8]:
test_input_1 = """ Title: Fruit and Vegetable Consumption and the Incidence of Hypertension in Three Prospective Cohort Studies. Abstract: Increased fruit and vegetable intake lowers blood pressure in short-term interventional studies. However, data on the association of long-term intake of fruits and vegetables with hypertension risk are scarce. We prospectively examined the independent association of whole fruit (excluding juices) and vegetable intake, as well as the change in consumption of whole fruits and vegetables, with incident hypertension in 3 large longitudinal cohort studies: Nurses' Health Study (n=62 175), Nurses' Health Study II (n=88 475), and Health Professionals Follow-up Study (n=36 803). We calculated hazard ratios and 95% confidence intervals for fruit and vegetable consumption while controlling for hypertension risk factors. Compared with participants whose consumption was ≤4 servings/week, the pooled hazard ratios among those whose intake was ≥4 servings/day were 0.92(0.87-0.97) for total whole fruit intake and 0.95(0.86-1.04) for total vegetable intake. Similarly, compared with participants who did not increase their fruit or vegetable consumption, the pooled hazard ratios for those whose intake increased by ≥7 servings/week were 0.94(0.90-0.97) for total whole fruit intake and 0.98(0.94-1.01) for total vegetable. Analyses of individual fruits and vegetables yielded different results. Consumption levels of ≥4 servings/week (as opposed to <1 serving/month) of broccoli, carrots, tofu or soybeans, raisins, and apples was associated with lower hypertension risk. In conclusion, our results suggest that greater long-term intake and increased consumption of whole fruits may reduce the risk of developing hypertension.
"""
test_input_2 = """ Title: Dietary intake of marine n-3 fatty acids, fish intake, and the risk of coronary disease among men. Abstract: Background: It has been hypothesized that a diet containing n-3 fatty acids from fish reduces the risk of coronary heart disease, but few large epidemiologic studies have examined this question.

Methods: In 1986, 44,895 male health professionals, 40 to 75 years of age, who were free of known cardiovascular disease completed detailed and validated dietary questionnaires as part of the Health Professionals Follow-up Study. During six years of follow-up, we documented 1543 coronary events in this group: 264 deaths from coronary disease, 547 nonfatal myocardial infarctions, and 732 coronary-artery bypass or angioplasty procedures.

Results: After controlling for age and several coronary risk factors, we observed no significant associations between dietary intake of n-3 fatty acids or fish intake and the risk of coronary disease. For men in the top fifth of the group in terms of intake of n-3 fatty acids (median, 0.58 g per day), the multivariate relative risk of coronary heart disease was 1.12 (95 percent confidence interval, 0.96 to 1.31), as compared with the men in the bottom fifth (median, 0.07 g per day). For men who consumed six or more servings of fish per week, as compared with those who consumed one serving per month or less, the multivariate relative risk of coronary disease was 1.14 (95 percent confidence interval, 0.86 to 1.51). The risk of death due to coronary disease among men who ate any amount of fish, as compared with those who ate no fish, was 0.74 (95 percent confidence interval, 0.44 to 1.23), but the risk did not decrease as fish consumption increased.

Conclusions: Although the possibility of residual confounding by unmeasured factors cannot be entirely excluded, these data suggest that increasing fish intake from one to two servings per week to five to six servings per week does not substantially reduce the risk of coronary heart disease among men who are initially free of cardiovascular disease.
"""

test_input_3 = """ Title: Relation of consumption of vitamin E, vitamin C, and carotenoids to risk for stroke among men in the United States

Abstract: Background: Antioxidants increase the resistance of low-density lipoprotein to oxidation and may thereby reduce risk for atherosclerosis.

Objective: To determine whether intake of vitamin E, vitamin C, or carotenoids predict risk for total or ischemic stroke.

Design: Prospective observational study.

Setting: The Health Professionals Follow-up Study.

Participants: 43,738 men 40 to 75 years of age who did not have cardiovascular disease or diabetes.

Measurements: Repeated and validated dietary assessments were done by using a self-administered 131-item food-frequency questionnaire, which included questions on dose and duration of vitamin supplement use. The follow-up period was 8 years.

Results: A total of 328 strokes occurred: 210 ischemic, 70 hemorrhagic, and 48 unclassified. After adjustment for age, smoking, hypertension, hypercholesterolemia, body mass index, physical activity, parental history of myocardial infarction, alcohol consumption, and total energy intake, the relative risk for ischemic stroke in the top quintile of vitamin E intake (median, 411 IU/d) compared with the bottom quintile (5.4 IU/d) was 1.18 (95% CI, 0.77 to 1.82). The relative risk for ischemic stroke in the top quintile of vitamin C intake (1167 mg/d) compared with the bottom quintile (95 mg/d) was 1.03 (CI, 0.66 to 1.59). Results for total stroke were similar. Associations of vitamin intake with hemorrhagic stroke were also nonsignificant, but the CIs were wide. Neither dose nor duration of vitamin E or vitamin C supplement use was related to risk for total or ischemic stroke. The relative risk for ischemic stroke was 1.16 (CI, 0.81 to 1.67) in men using 250 IU or more of vitamin E supplementation per day compared with men who used no vitamin E supplements and was 0.93 (CI, 0.60 to 1.45) in men using 700 mg or more of vitamin C supplementation per day compared with men who used no vitamin C supplements. A significant inverse relation between lutein intake and risk for ischemic stroke was seen but was not independent of other dietary factors.

Conclusions: Vitamin E and vitamin C supplements and specific carotenoids did not seem to substantially reduce risk for stroke in this cohort. Modest effects, however, cannot be excluded.
"""

In [9]:
# Define the prompt for step 1
prompt_step1 = """
From the following given title and abstract, extract all studied exposures and cardiovascular-related outcomes separately. Ignore any other information.

Extract the following information:

1. "exposures": a list of all studied exposures
2. "outcomes": a list of all studied cardiovascular-related outcomes. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.

Return the extracted information in a JSON dictionary.

Example input:
""
Title: "Dietary fat and risk of coronary heart disease in men: cohort follow up study in the United States"

Abstract: "Objective: To examine the association between fat intake and the incidence of coronary heart disease in men of middle age and older. Design: Cohort questionnaire study of men followed up for six years from 1986. Setting: The health professionals follow up study in the United States. Subjects: 43 757 health professionals aged 40 to 75 years free of diagnosed cardiovascular disease or diabetes in 1986. Main outcome measure: Incidence of acute myocardial infarction or coronary death. Results: During follow up 734 coronary events were documented, including 505 non-fatal myocardial infarctions and 229 deaths. After age and several coronary risk factors were controlled for significant positive associations were observed between intake of saturated fat and risk of coronary disease. For men in the top versus the lowest fifth of saturated fat intake (median = 14.8% v 5.7% of energy) the multivariate relative risk for myocardial infarction was 1.22 (95% confidence interval 0.96 to 1.56) and for fatal coronary heart disease was 2.21 (1.38 to 3.54). After adjustment for intake of fibre the risks were 0.96 (0.73 to 1.27) and 1.72 (1.01 to 2.90), respectively. Positive associations between intake of cholesterol and risk of coronary heart disease were similarly attenuated after adjustment for fibre intake. Intake of linolenic acid was inversely associated with risk of myocardial infarction; this association became significant only after adjustment for non-dietary risk factors and was strengthened after adjustment for total fat intake (relative risk 0.41 for a 1% increase in energy, P for trend < 0.01)."
Conclusions: These data do not support the strong association between intake of saturated fat and risk of coronary heart disease suggested by international comparisons. They are compatible, however, with the hypotheses that saturated fat and cholesterol intakes affect the risk of coronary heart disease as predicted by their effects on blood cholesterol concentration. They also support a specific preventive effect of linolenic acid intake.
""

Example output:
{
    "exposures": ["fat intake", "saturated fat", "cholesterol", "linolenic acid"],
    "outcomes": ["CHD", "MI"]
}

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ```in your reponse
"""

prompt_step2_template = """
From the following given title and abstract, extract the associations between the given dietary exposures and cardiovascular complications, such as coronary heart disease, stroke, and hypertension.

Use the provided entities as references.

fyi: CHD refers to non-fatal MI and fatal coronary heart disease; also, coronary artery disease (CAD) refers to non-fatal MI and fatal coronary artery disease. Cardiovascular disease (CVD) is defined as a composite of coronary artery disease and non-fatal or fatal stroke.

Extract the following information:

1. "exposure": the studied exposure
2. "label": standardized exposure concept
3. "category": the category of exposure based on 9 categories: 'Food/Diet', 'Lipid', 'Mineral', 'Vitamin', 'Carb', 'Alcohol', 'Fiber', 'Other', 'Protein'
4. "outcome_phenotype": the studied outcome, which is a cardiovascular-related disease such as CHD, stroke, etc. Only use abbreviations for CHD, MI, and CVD. Return the full name for others.
5. "association": the association between exposure and outcome, which is either positive, negative, or no association
6. "effect_size": the effect size of the association. Only return for positive and negative associations. Return "not found" for no association (output number, do not output percentage)

Return the extracted information in a list of flat JSON dictionaries of each result.

Example input entities:
{{
    "exposures": ["fat intake", "saturated fat", "cholesterol", "linolenic acid"],
    "outcomes": ["CHD", "MI"]
}}

Example input title and abstract:
""
Title: "Dietary fat and risk of coronary heart disease in men: cohort follow up study in the United States"

Abstract: "Objective: To examine the association between fat intake and the incidence of coronary heart disease in men of middle age and older. Design: Cohort questionnaire study of men followed up for six years from 1986. Setting: The health professionals follow up study in the United States. Subjects: 43 757 health professionals aged 40 to 75 years free of diagnosed cardiovascular disease or diabetes in 1986. Main outcome measure: Incidence of acute myocardial infarction or coronary death. Results: During follow up 734 coronary events were documented, including 505 non-fatal myocardial infarctions and 229 deaths. After age and several coronary risk factors were controlled for significant positive associations were observed between intake of saturated fat and risk of coronary disease. For men in the top versus the lowest fifth of saturated fat intake (median = 14.8% v 5.7% of energy) the multivariate relative risk for myocardial infarction was 1.22 (95% confidence interval 0.96 to 1.56) and for fatal coronary heart disease was 2.21 (1.38 to 3.54). After adjustment for intake of fibre the risks were 0.96 (0.73 to 1.27) and 1.72 (1.01 to 2.90), respectively. Positive associations between intake of cholesterol and risk of coronary heart disease were similarly attenuated after adjustment for fibre intake. Intake of linolenic acid was inversely associated with risk of myocardial infarction; this association became significant only after adjustment for non-dietary risk factors and was strengthened after adjustment for total fat intake (relative risk 0.41 for a 1% increase in energy, P for trend < 0.01)."
Conclusions: These data do not support the strong association between intake of saturated fat and risk of coronary heart disease suggested by international comparisons. They are compatible, however, with the hypotheses that saturated fat and cholesterol intakes affect the risk of coronary heart disease as predicted by their effects on blood cholesterol concentration. They also support a specific preventive effect of linolenic acid intake.
""

Example output:
[
    {{"exposure": "linolenic acid", "label": "Linolenic acid", "category": "Lipid", "outcome_phenotype": "MI", "association": "negative", "effect_size": "0.41"}},
    {{"exposure": "saturated fat", "label": "SFA", "category": "Lipid", "outcome_phenotype": "CHD", "association": "positive", "effect_size": "2.21"}},
    {{"exposure": "saturated fat", "label": "SFA", "category": "Lipid", "outcome_phenotype": "MI", "association": "positive", "effect_size": "1.22"}},
    {{"exposure": "saturated fat adjusted for fiber", "label": "SFA adjusted for fiber", "category": "Lipid", "outcome_phenotype": "CHD", "association": "no", "effect_size": "not found"}},
    {{"exposure": "saturated fat adjusted for fiber", "label": "SFA adjusted for fiber", "category": "Lipid", "outcome_phenotype": "MI", "association": "no", "effect_size": "not found"}}
]

Only output in JSON format, do not include any explanations or additional text in your response. no ```json ``` in your reponse
"""

In [ ]:
MODEL="deepseek-chat"
client = OpenAI(api_key='YOUR_DEEPSEEK_API_KEY',base_url="https://api.deepseek.com")

# Function to perform the two-step extraction
def two_step_extraction(text):
    # Step 1: Extract exposures and outcomes
    completion_step1 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
            {"role": "user", "content": prompt_step1},
            {"role": "user", "content": text}
        ]
    )
    entities_result = completion_step1.choices[0].message.content
    entities_result = entities_result.strip().removeprefix("```json").removesuffix("```").strip()
    entities_dict = json.loads(entities_result)
    entities_json = json.dumps(entities_dict, indent=2)

    # Prepare step 2 prompt
    prompt_step2 = prompt_step2_template.format(entities=entities_json)

    # Step 2: Extract associations using the entities
    completion_step2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
            {"role": "user", "content": prompt_step2},
            {"role": "user", "content": f"extracted entities: {entities_json}"},
            {"role": "user", "content": f"title and abstract: {text}"}
        ]
    )
    associations_result = completion_step2.choices[0].message.content

    return associations_result

# Worker function to process each row
def worker(input_queue, output_queue, text_column, progress_bar):
    while True:
        index, row = input_queue.get()
        if index is None:
            break
        try:
            output = two_step_extraction(row[text_column])
            extracted_results = literal_eval(output)
            for res in extracted_results:
                res['pmid'] = row['pmid']
                output_queue.put(res)
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            print(f"Input text: {row[text_column][:500]}...")  # Print the first 500 characters of the text
            print(f"Output: {output}")  # Print the output if available
        finally:
            input_queue.task_done()
            progress_bar.update(1)

# Main function to compile extracted results
def compile_extracted_results(input_df, text_column):
    results = []
    input_queue = Queue()
    output_queue = Queue()

    # Progress bar for processing
    progress_bar = tqdm(total=input_df.shape[0], desc="Processing abstracts")

    # Start worker threads
    for _ in range(8):
        threading.Thread(target=worker, args=(input_queue, output_queue, text_column, progress_bar), daemon=True).start()

    # Enqueue rows for processing
    for index, row in input_df.iterrows():
        input_queue.put((index, row))

    # Wait for all tasks to be processed
    input_queue.join()

    # Collect results from output queue
    while not output_queue.empty():
        results.append(output_queue.get())

    # Close progress bar
    progress_bar.close()

    results_df = pd.json_normalize(results)
    return results_df

In [15]:
# Step 1: Extract exposures and outcomes
completion_step1 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me extract structured information from free texts of article titles and abstracts"},
    {"role": "user", "content": prompt_step1},
    {"role": "user", "content": test_input_3}
  ]
)
entities_result = completion_step1.choices[0].message.content
print("Entities extracted:", entities_result)

# Parse the entities result to ensure it's in proper JSON format
entities_dict = json.loads(entities_result)
entities_json = json.dumps(entities_dict, indent=2)

# Prepare prompt for step 2
prompt_step2 = prompt_step2_template#.format(entities=entities_json)

# Step 2: Extract associations using the entities
completion_step2 = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are an evidence-based medicine annotator in dietary and cardiovascular diseases, help me infer associations from given entities and titles and abstracts"},
    {"role": "user", "content": prompt_step2},
    {"role": "user", "content": f"extracted entities: {entities_json}"},
    {"role": "user", "content": f"title and abstract: {test_input_3}"}
  ]
)
associations_result = completion_step2.choices[0].message.content
print("Associations extracted:", associations_result)

Entities extracted: {
    "exposures": ["vitamin E", "vitamin C", "carotenoids", "lutein"],
    "outcomes": ["stroke", "ischemic stroke", "hemorrhagic stroke"]
}
Associations extracted: [
{"exposure": "vitamin E", "label": "Vitamin E", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
{"exposure": "vitamin C", "label": "Vitamin C", "category": "Vitamin", "outcome_phenotype": "ischemic stroke", "association": "no", "effect_size": "not found"},
{"exposure": "lutein", "label": "Lutein", "category": "Other", "outcome_phenotype": "ischemic stroke", "association": "negative", "effect_size": "not found"}
]


# evaluation #1

In [2]:
gold_df_1 = pd.read_csv('../data/vali_data/nc_vali_df.csv')
pred_df_1 = pd.read_csv('../data/vali_data/result_df_2_deepseek_rev.csv')
#pred_df=result_df_2_deepseek
print(gold_df_1['association'].unique(),pred_df_1['association'].unique())

['no' 'negative' 'positive'] ['no' 'negative' 'positive']


## entities

In [3]:
#gold to extracted

gold_standard = gold_df_1[['pmid', 'exposure', 'outcome_phenotype']].astype(str)
extracted_info = pred_df_1[['pmid', 'exposure', 'outcome_phenotype']].astype(str)

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold_df, pred_df, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold_df['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold_df[gold_df['pmid'] == pmid][column].unique().tolist()
        pred_texts = pred_df[pred_df['pmid'] == pmid][column].unique().tolist()
        if gold_texts and pred_texts:
            for gold_text in gold_texts:
                P, R, F1 = score([gold_text], [pred_texts], lang='en', model_type='microsoft/deberta-xlarge-mnli',device='cpu')
                results['P'].append(P.mean().item())
                results['R'].append(R.mean().item())
                results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for exposure and outcome
bertscore_exposure_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(gold_standard, extracted_info, 'outcome_phenotype')

# Aggregate results
def aggregate_results(results):
    if not results['P']:  # Check if the list is empty
        return 0, 0, 0
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (outcome_phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 76/76 [06:15<00:00,  4.94s/it]

BERTScore for NER (Exposure): (0.7961453712521456, 0.8053921960676328, 0.794914376659271)
BERTScore for NER (outcome_phenotype): (0.8709966009284588, 0.8836332740301781, 0.8733203661568621)


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data

# Filter out columns for the comparison
gold_standard = gold_df_1[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)
extracted_info = pred_df_1[['pmid', 'exposure', 'outcome_phenotype', 'association']].drop_duplicates()#.astype(str).sort_values('pmid').head(20)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_phenotype_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_phenotype_gold'] == gold_texts_out[i])]['association_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_phenotype_pred'] == pred_texts_out[i])]['association_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_phenotype_gold', 'outcome_phenotype_pred', 'association_gold', 'association_pred'])

# Calculate accuracy
accuracy = accuracy_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'])

# Calculate precision, recall, and F1-score
precision = precision_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
recall = recall_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')
f1 = f1_score(matching_pairs_df['association_gold'], matching_pairs_df['association_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score
print("Accuracy of association between matched exposure and outcome_phenotype pairs:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Identify and print falsely predicted associations
falsely_predicted = matching_pairs_df[matching_pairs_df['association_gold'] != matching_pairs_df['association_pred']]
print("Falsely predicted associations:")
print(falsely_predicted.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

100%|██████████| 76/76 [10:40<00:00,  8.43s/it]

Accuracy of association between matched exposure and outcome_phenotype pairs: 0.835820895522388
Precision: 0.7945458643227407
Recall: 0.8476264189886481
F1-score: 0.8102325928412886
Falsely predicted associations:
|     |     pmid | exposure_gold                      | exposure_pred                      | outcome_phenotype_gold    | outcome_phenotype_pred    | association_gold   | association_pred   |
|----:|---------:|:-----------------------------------|:-----------------------------------|:--------------------------|:--------------------------|:-------------------|:-------------------|
|  23 | 14980987 | cereal fiber                       | cereal fiber                       | CHD                       | CHD                       | no                 | negative           |
|  24 | 14980987 | cereal fiber                       | cereal fiber                       | CHD                       | CHD                       | no                 | negative           |
|  29 | 15585762 | sup

In [18]:
# Extract the metrics
relation_accuracy = round(accuracy * 100)
relation_recall = round(recall * 100)

print("指标3.1 因果关系抽取准确率/召回率 数据集1.a ")
print("关系数量：",(gold_df_1.shape))
print(f"  准确率: {relation_accuracy}%")
print(f"  召回率: {relation_recall}%")

指标3.1 因果关系抽取准确率/召回率 数据集1.a 
关系数量： (249, 8)
  准确率: 84%
  召回率: 85%


# evaluation #2

In [6]:
gold_df_2 = pd.read_csv('../data/vali_data/amia_vali_df_new.csv')
pred_df_2 = pd.read_csv('../data/vali_data/amia_result_df_2_deepseek.csv')

print(gold_df_2['pmid'].nunique(),pred_df_2['pmid'].nunique())
gold_df_2 = gold_df_2[gold_df_2['pmid'].isin(pred_df_2['pmid'])]
print(gold_df_2['pmid'].nunique(),pred_df_2['pmid'].nunique())


print(gold_df_2['direction'].unique(),pred_df_2['direction'].unique())
print(gold_df_2['significance'].unique(),pred_df_2['significance'].unique())

90 89
89 89
['decrease' 'increase' 'not found'] ['increase' 'decrease' 'not found']
['positive' 'negative' 'not found'] ['positive' 'negative' 'not found']


In [7]:
from sklearn.metrics import precision_recall_fscore_support
from bert_score import score
import pandas as pd

# Load the data

gold_standard = gold_df_2[['pmid', 'exposure', 'outcome']].astype(str)
extracted_info = pred_df_2[['pmid', 'exposure', 'outcome']].astype(str)

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Function to calculate BERTScore for a specific column for each pmid
def calculate_bertscore_per_pmid(gold, pred, column):
    results = {'P': [], 'R': [], 'F1': []}
    pmids = gold['pmid'].unique()
    for pmid in tqdm(pmids):
        gold_texts = gold[gold['pmid'] == pmid][column + '_gold'].tolist()
        pred_texts = pred[pred['pmid'] == pmid][column + '_pred'].tolist()
        if gold_texts and pred_texts:
            P, R, F1 = score(pred_texts, gold_texts, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            results['P'].append(P.mean().item())
            results['R'].append(R.mean().item())
            results['F1'].append(F1.mean().item())
    return results

# Calculate BERTScore for NER and RE
bertscore_exposure_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'exposure')
bertscore_outcome_results = calculate_bertscore_per_pmid(merged_df, merged_df, 'outcome')

# Aggregate results
def aggregate_results(results):
    P_avg = sum(results['P']) / len(results['P'])
    R_avg = sum(results['R']) / len(results['R'])
    F1_avg = sum(results['F1']) / len(results['F1'])
    return P_avg, R_avg, F1_avg

bertscore_ner_exposure_avg = aggregate_results(bertscore_exposure_results)
bertscore_ner_outcome_avg = aggregate_results(bertscore_outcome_results)

# Print the results
print("BERTScore for NER (Exposure):", bertscore_ner_exposure_avg)
print("BERTScore for NER (Outcome Phenotype):", bertscore_ner_outcome_avg)

100%|██████████| 89/89 [06:08<00:00,  4.14s/it]

BERTScore for NER (Exposure): (0.8266496557867928, 0.8106519141223993, 0.8154910786098308)
BERTScore for NER (Outcome Phenotype): (0.6582437981380506, 0.6391207497106509, 0.6453132526425833)


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from bert_score import score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the data
# Assuming gold_df and pred_df are your DataFrames

# Filter out columns for the comparison
gold_standard = gold_df_2[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()
extracted_info = pred_df_2[['pmid', 'exposure', 'outcome', 'direction', 'significance']].drop_duplicates()

# Merge on pmid
merged_df = pd.merge(gold_standard, extracted_info, on='pmid', suffixes=('_gold', '_pred'))

# Ensure all text columns are strings and handle NaN values
merged_df['exposure_gold'] = merged_df['exposure_gold'].astype(str).fillna('')
merged_df['exposure_pred'] = merged_df['exposure_pred'].astype(str).fillna('')
merged_df['outcome_gold'] = merged_df['outcome_gold'].astype(str).fillna('')
merged_df['outcome_pred'] = merged_df['outcome_pred'].astype(str).fillna('')

# Function to calculate BERTScore and find matching pairs within each pmid
def find_matching_pairs(merged_df, threshold=0.8):
    matching_pairs = []
    pmids = merged_df['pmid'].unique()

    for pmid in tqdm(pmids):
        gold_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_gold'].tolist()
        pred_texts_exp = merged_df[merged_df['pmid'] == pmid]['exposure_pred'].tolist()
        gold_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_gold'].tolist()
        pred_texts_out = merged_df[merged_df['pmid'] == pmid]['outcome_pred'].tolist()

        if gold_texts_exp and pred_texts_exp and gold_texts_out and pred_texts_out:
            P_exp, R_exp, F1_exp = score(pred_texts_exp, gold_texts_exp, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            P_out, R_out, F1_out = score(pred_texts_out, gold_texts_out, lang='en', model_type='microsoft/deberta-xlarge-mnli')

            for i in range(len(pred_texts_exp)):
                if F1_exp[i].item() > threshold and F1_out[i].item() > threshold:
                    matching_pairs.append((
                        pmid,
                        gold_texts_exp[i],
                        pred_texts_exp[i],
                        gold_texts_out[i],
                        pred_texts_out[i],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['direction_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['direction_pred'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_gold'] == gold_texts_exp[i]) & (merged_df['outcome_gold'] == gold_texts_out[i])]['significance_gold'].values[0],
                        merged_df[(merged_df['pmid'] == pmid) & (merged_df['exposure_pred'] == pred_texts_exp[i]) & (merged_df['outcome_pred'] == pred_texts_out[i])]['significance_pred'].values[0]
                    ))

    return matching_pairs

# Find matching pairs
matching_pairs = find_matching_pairs(merged_df, threshold=0.8)

# Convert matching pairs to DataFrame
matching_pairs_df = pd.DataFrame(matching_pairs, columns=['pmid', 'exposure_gold', 'exposure_pred', 'outcome_gold', 'outcome_pred', 'direction_gold', 'direction_pred', 'significance_gold', 'significance_pred'])

# Calculate accuracy for direction
accuracy_direction = accuracy_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'])
# Calculate precision, recall, and F1-score for direction
precision_direction = precision_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
recall_direction = recall_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')
f1_direction = f1_score(matching_pairs_df['direction_gold'], matching_pairs_df['direction_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for direction
print("Accuracy of direction:", accuracy_direction)
print("Precision of direction:", precision_direction)
print("Recall of direction:", recall_direction)
print("F1-score of direction:", f1_direction)

# Calculate accuracy for significance
accuracy_significance = accuracy_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'])
# Calculate precision, recall, and F1-score for significance
precision_significance = precision_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
recall_significance = recall_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')
f1_significance = f1_score(matching_pairs_df['significance_gold'], matching_pairs_df['significance_pred'], average='macro')

# Print the accuracy, precision, recall, and F1-score for significance
print("Accuracy of significance:", accuracy_significance)
print("Precision of significance:", precision_significance)
print("Recall of significance:", recall_significance)
print("F1-score of significance:", f1_significance)

# Identify and print falsely predicted associations for direction
falsely_predicted_direction = matching_pairs_df[matching_pairs_df['direction_gold'] != matching_pairs_df['direction_pred']]
print("Falsely predicted associations (direction):")
print(falsely_predicted_direction.to_markdown())

# Identify and print falsely predicted associations for significance
falsely_predicted_significance = matching_pairs_df[matching_pairs_df['significance_gold'] != matching_pairs_df['significance_pred']]
print("Falsely predicted associations (significance):")
print(falsely_predicted_significance.to_markdown())

# Print the number of predicted pairs, gold pairs, and matched pairs
print(f"Number of gold pairs: {len(gold_standard)}")
print(f"Number of predicted pairs: {len(extracted_info)}")
print(f"Number of matched pairs: {len(matching_pairs_df)}")

# Print matched pairs
print("Matched pairs:")
print(matching_pairs_df.to_markdown())

  0%|          | 0/89 [00:00<?, ?it/s]

100%|██████████| 89/89 [11:51<00:00,  8.00s/it]

Accuracy of direction: 0.8434782608695652
Precision of direction: 0.8228713070275341
Recall of direction: 0.824385996517144
F1-score of direction: 0.8207984833277573
Accuracy of significance: 0.9565217391304348
Precision of significance: 0.9382716049382717
Recall of significance: 0.9818840579710145
F1-score of significance: 0.9566754075932048
Falsely predicted associations (direction):
|     |     pmid | exposure_gold                                                                        | exposure_pred                                                                        | outcome_gold                                    | outcome_pred                             | direction_gold   | direction_pred   | significance_gold   | significance_pred   |
|----:|---------:|:-------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------|:------------------------------------------------|:

In [17]:
# Extract the metrics
relation_accuracy_2 = round(accuracy_direction * 100)
relation_recall_2 = round(recall_direction * 100)

print("指标3.1 因果关系抽取准确率/召回率 数据集1.b")
print("关系数量：",len(gold_df_2))
print(f"  准确率: {relation_accuracy_2}%")
print(f"  召回率: {relation_recall_2}%")

指标3.1 因果关系抽取准确率/召回率 数据集1.b
关系数量： 294
  准确率: 84%
  召回率: 82%


In [16]:
print('3.1 因果关系抽取准确率/召回率')
print('dataset 1.a 状态: ')
print(f"  准确率: {relation_accuracy}%")
print(f"  召回率: {relation_recall}%")

print('dataset 1.b 状态: ')
print(f"  准确率: {relation_accuracy_2}%")
print(f"  召回率: {relation_recall_2}%")


print('''
dataset 2：因果关系抽取指标如下 见data2文件夹:
准确率 (Precision): 0.8694
召回率 (Recall): 0.9797
F1分数 (F1-Score): 0.9213''')

3.1 因果关系抽取准确率/召回率
dataset 1.a 状态: 
  准确率: 84%
  召回率: 85%
dataset 1.b 状态: 
  准确率: 84%
  召回率: 82%

dataset 2：因果关系抽取指标如下 见data2文件夹:
准确率 (Precision): 0.8694
召回率 (Recall): 0.9797
F1分数 (F1-Score): 0.9213


In [19]:
print('''3.1 因果关系抽取准确率/召回率
总体测试集数量:  3101

最高准确率、召回率、F1:
准确率 (Precision): 0.8694
召回率 (Recall): 0.9797
F1分数 (F1-Score): 0.9213
''')

3.1 因果关系抽取准确率/召回率
总体测试集数量:  3101

最高准确率、召回率、F1:
准确率 (Precision): 0.8694
召回率 (Recall): 0.9797
F1分数 (F1-Score): 0.9213

